# LIMS result pushdown (cognite-pygen-spark 0.4.1, cognite-databricks 0.4.1)

Queries the synthetic `LimsResult` view on cognite-cogsail. The view has the 40-column LIMS result shape.
100,000 nodes are already in CDF:

- `dm_dom_lims_result` is the data model space (not an instance space)
- `inst_lims_result_lab_a` has 60,000 nodes (`result_000001` .. `result_060000`)
- `inst_lims_result_lab_b` has 40,000 nodes (`result_060001` .. `result_100000`)

Seed them from a pygen-spark checkout before this notebook if the CDF check below fails:

```
uv run python -m tests.test_live.lims_result
```

The notebook installs the released 0.4.1 packages, writes `base_url` into the secret scope, registers the view, and runs the population queries against that view. Paste nothing into source control after the credentials TOML path is set. The notebook prints the project and `base_url` only.


In [ ]:
%pip install "cognite-pygen-spark==0.4.1" "cognite-databricks==0.4.1"


In [ ]:
%restart_python


In [ ]:
import cognite.databricks
import cognite.pygen_spark

assert cognite.databricks.__version__ == "0.4.1", cognite.databricks.__version__
assert cognite.pygen_spark.__version__ == "0.4.1", cognite.pygen_spark.__version__
print(cognite.databricks.__version__, cognite.pygen_spark.__version__)


In [ ]:
import tomllib
from pathlib import Path

from cognite.client import CogniteClient
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthClientCredentials
from databricks.sdk import WorkspaceClient

# Workspace path of the cogsail credentials TOML. Do not commit a filled-in path if the file is local-only.
TOML_PATH = ""

CATALOG = "f0connectortest"
SECRET_SCOPE = "cdf_dm_dom_lims_result_limsresult_dom"
DM_SPACE = "dm_dom_lims_result"
MODEL_EXTERNAL_ID = "LimsResult_DOM"
MODEL_VERSION = "v1"
VIEW = f"{CATALOG}.{DM_SPACE}_{MODEL_EXTERNAL_ID.lower()}_{MODEL_VERSION}.LimsResult"
LAB_A = "inst_lims_result_lab_a"
LAB_B = "inst_lims_result_lab_b"

if not TOML_PATH:
    raise ValueError("Set TOML_PATH to the cogsail credentials TOML")

with Path(TOML_PATH).open("rb") as fh:
    data = tomllib.load(fh)
cognite = data["cognite"]
runtime = data.get("fn_btp_hierarchy_runtime") or {}
client_id = runtime.get("client_id") or cognite.get("client_id")
client_secret = runtime.get("client_secret") or cognite.get("client_secret")
if not client_id or not client_secret:
    triggers = data.get("hierarchy_workflow_triggers") or {}
    secrets = data.get("hierarchy_workflow_trigger_secrets") or {}
    client_id = client_id or triggers.get("ingest_trigger_client_id")
    client_secret = client_secret or secrets.get("ingest_trigger_client_secret")
if not client_id or not client_secret:
    raise ValueError("TOML is missing client_id / client_secret")
tenant_id = cognite.get("tenant_id") or cognite.get("idp_tenant_id")
if not tenant_id:
    raise ValueError("TOML [cognite] is missing tenant_id")

base_url = f"https://{cognite['cdf_cluster']}.cognitedata.com"
client = CogniteClient(
    ClientConfig(
        client_name="lims-result-0.4.1",
        project=cognite["project"],
        credentials=OAuthClientCredentials(
            token_url=f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
            client_id=client_id,
            client_secret=client_secret,
            scopes=[f"{base_url}/.default"],
        ),
        base_url=base_url,
    )
)
workspace_client = WorkspaceClient()
print("project:", client.config.project)
print("base_url:", client.config.base_url)


## CDF population

These counts come from `instances/aggregate`, the same payloads the UDTF sends. The aggregate index can lag a new seed by several minutes.


In [ ]:
import time

from cognite.client import data_modeling as dm
from cognite.pygen_spark.filters import (
    AggregateMetric,
    AggregateRequestSpec,
    FilterSpec,
    build_aggregate_payload,
    build_filter_json,
    parse_aggregate_count,
    parse_metric_aggregates,
)

VIEW_ID = dm.ViewId(space=DM_SPACE, external_id="LimsResult", version=MODEL_VERSION)


def cdf_count(instance_space=None, **kwargs) -> int:
    spec = FilterSpec(
        view_space=VIEW_ID.space,
        view_external_id=VIEW_ID.external_id,
        view_version=VIEW_ID.version,
        instance_space=instance_space,
        **kwargs,
    )
    payload = build_aggregate_payload(
        AggregateRequestSpec(
            view_space=VIEW_ID.space,
            view_external_id=VIEW_ID.external_id,
            view_version=VIEW_ID.version,
            aggregates=[AggregateMetric(fn="count", property="externalId")],
            filter_json=build_filter_json(spec),
        )
    )
    url = f"/api/v1/projects/{client.config.project}/models/instances/aggregate"
    return parse_aggregate_count(client.post(url, json=payload).json())


deadline = time.monotonic() + 900
lab_a = cdf_count(LAB_A)
lab_b = cdf_count(LAB_B)
while (lab_a, lab_b) != (60_000, 40_000) and time.monotonic() < deadline:
    time.sleep(5)
    lab_a = cdf_count(LAB_A)
    lab_b = cdf_count(LAB_B)
print("lab A", lab_a, "lab B", lab_b)
assert (lab_a, lab_b) == (60_000, 40_000)
assert cdf_count([LAB_A, LAB_B]) == 100_000
assert cdf_count(DM_SPACE) == 0
assert cdf_count(property_filters={"componentName": "MOISTURE_CONTENT"}) == 10_000
assert cdf_count(property_filters={"isOutOfSpec": True}) == 1_000

fixture = client.data_modeling.instances.retrieve(nodes=(LAB_A, "result_000001"), sources=[VIEW_ID])
assert fixture.nodes, "Node result_000001 not found in CDF"
props = fixture.nodes[0].properties[VIEW_ID]
assert props["numericValue"] == 12.5
assert props["componentName"] == "MOISTURE_CONTENT"
print("CDF population matches the seed")


## Register the 0.4.1 view

`set_cdf_credentials` writes `base_url` with the other credential keys and replaces a host already stored in the scope.


In [ ]:
from cognite.client.data_classes.data_modeling.ids import DataModelId
from cognite.databricks import generate_udtf_notebook

generator = generate_udtf_notebook(
    DataModelId(space=DM_SPACE, external_id=MODEL_EXTERNAL_ID, version=MODEL_VERSION),
    client,
    workspace_client=workspace_client,
    catalog=CATALOG,
    output_dir="/local_disk0/tmp/pygen_udtf",
)
generator.secret_helper.set_cdf_credentials(
    scope_name=SECRET_SCOPE,
    project=client.config.project,
    cdf_cluster=cognite["cdf_cluster"],
    client_id=client_id,
    client_secret=client_secret,
    tenant_id=tenant_id,
    base_url=client.config.base_url,
)
secret_keys = sorted(item.key for item in workspace_client.secrets.list_secrets(scope=SECRET_SCOPE))
print("secret keys:", secret_keys)
assert "base_url" in secret_keys

udtfs = generator.register_udtfs(secret_scope=SECRET_SCOPE, if_exists="replace")
print("registered UDTFs:", udtfs.total_count)
views = generator.register_views(secret_scope=SECRET_SCOPE, if_exists="replace")
print("registered views:", [item.view_name for item in views.registered_views])


## Pushdown query times

Same two questions as the 0.4.1 installation notebook: a count, and a selective `SELECT *`.

`rewrite_query` sends the count as a CDF aggregate and the filter as a CDF list filter. The cell prints the SQL result, then the seconds spent collecting it. A query on the registered view applies `WHERE` only after the UDTF returns.


In [ ]:
import time


def measure(name: str, analyst_sql: str):
    """Run the rewritten UDTF call and print its SQL result."""
    print(f"\n=== {name} ===")
    print(analyst_sql.strip())
    rewritten = generator.rewrite_query(analyst_sql, secret_scope=SECRET_SCOPE)
    if rewritten is None:
        raise RuntimeError(name + " was not pushed down")
    frame = spark.sql(rewritten)
    started = time.perf_counter()
    rows = frame.collect()
    elapsed = time.perf_counter() - started
    result = spark.createDataFrame(rows, schema=frame.schema)
    result.show(20, truncate=False)
    display(result.limit(20))
    print(f"{elapsed:.2f}s  {len(rows)} row(s)")
    return rows, elapsed


count_rows, count_seconds = measure("count", f"SELECT count(*) AS n FROM {VIEW}")
print("count value:", count_rows[0]["count_externalId"])
assert count_rows[0]["count_externalId"] == "100000"

fixture_rows, filter_seconds = measure(
    "filtered select",
    f"""
    SELECT *
    FROM {VIEW}
    WHERE external_id = 'result_000001'
    """,
)
assert len(fixture_rows) == 1
assert fixture_rows[0]["componentName"] == "MOISTURE_CONTENT"
assert fixture_rows[0]["numericValue"] == 12.5
print(f"count {count_seconds:.2f}s, filter {filter_seconds:.2f}s")
